In [ ]:
import requests

CKAN_URL = "https://catalogodatos.cnmc.es"
headers = {"User-Agent": "Application"}

query = 'Precios diarios provinciales - 2024 -'  # tal cual lo indica la web

url = f"{CKAN_URL}/api/3/action/package_search"
params = {"q": f'"{query}"'}  # comillas para buscar la frase exacta

r = requests.get(url, headers=headers, params=params, timeout=30)
r.raise_for_status()

datos_catalogo = r.json()

# Por seguridad: ver cuántos resultados trajo
print("success:", datos_catalogo.get("success"))
print("count:", datos_catalogo["result"]["count"])

# Sacar el resource_id como dice la web: resources[0].id
id_recurso = datos_catalogo["result"]["results"][0]["resources"][0]["id"]
print("resource_id:", id_recurso)


success: True
count: 1
resource_id: 141fdb3b-7c56-4eed-bf8d-bee56e577aa6


In [8]:
import requests
from typing import Dict, Tuple, Any

CKAN_URL = "https://catalogodatos.cnmc.es"
HEADERS = {"User-Agent": "Application"}

MIN_YEAR = 2016
MAX_YEAR = 2025

def _package_search(query: str, rows: int = 20) -> Dict[str, Any]:
    """Llama a CKAN package_search y devuelve el JSON (ya parseado)."""
    url = f"{CKAN_URL}/api/3/action/package_search"
    params = {"q": f'"{query}"', "rows": rows}  # comillas => frase exacta
    r = requests.get(url, headers=HEADERS, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def _pick_resource_id(results: list) -> str:
    """
    Elige el resource_id más adecuado.
    Regla robusta: preferir el recurso con datastore_active=True.
    Si no existe, lanza error.
    """
    if not results:
        raise ValueError("No hubo resultados en package_search.")

    # Tomamos el primer dataset (mejor match) pero elegimos el resource correcto.
    dataset = results[0]
    resources = dataset.get("resources") or []

    # Preferir el resource que realmente está en DataStore (compatible con datastore_search)
    for res in resources:
        if res.get("datastore_active") is True and res.get("id"):
            return res["id"]

    raise ValueError("Dataset encontrado, pero ningún resource tiene datastore_active=True.")

def build_resource_map(start_year: int, end_year: int) -> Tuple[Dict[int, str], Dict[int, str]]:
    """
    Construye un diccionario {año: resource_id} para el dataset:
    'Precios diarios provinciales - {year} -'
    entre start_year y end_year (incluidos), validando rango [2016, 2025].

    Si un año falla, lo registra en errores y continúa.
    Devuelve: (ids_por_anio, errores_por_anio)
    """
    # Validaciones del rango
    if not (MIN_YEAR <= start_year <= MAX_YEAR):
        raise ValueError(f"start_year debe estar entre {MIN_YEAR} y {MAX_YEAR}. Recibido: {start_year}")
    if not (MIN_YEAR <= end_year <= MAX_YEAR):
        raise ValueError(f"end_year debe estar entre {MIN_YEAR} y {MAX_YEAR}. Recibido: {end_year}")
    if start_year > end_year:
        raise ValueError(f"start_year ({start_year}) no puede ser mayor que end_year ({end_year}).")

    ids_por_anio: Dict[int, str] = {}
    errores_por_anio: Dict[int, str] = {}

    for year in range(start_year, end_year + 1):
        query = f"Precios diarios provinciales - {year} -"

        try:
            datos_catalogo = _package_search(query=query, rows=20)

            # Comprobación básica de éxito CKAN
            if not datos_catalogo.get("success", False):
                errores_por_anio[year] = f"CKAN success=False. Respuesta: {datos_catalogo}"
                print(f"[{year}] ERROR: CKAN devolvió success=False")
                continue

            results = datos_catalogo.get("result", {}).get("results", [])
            count = datos_catalogo.get("result", {}).get("count", 0)

            if count == 0 or not results:
                errores_por_anio[year] = "No se encontraron datasets con esa query."
                print(f"[{year}] ERROR: No se encontraron resultados (count=0).")
                continue

            resource_id = _pick_resource_id(results)
            ids_por_anio[year] = resource_id
            print(f"[{year}] OK -> resource_id: {resource_id}")

        except requests.exceptions.RequestException as e:
            errores_por_anio[year] = f"Error HTTP/red: {repr(e)}"
            print(f"[{year}] ERROR HTTP/red: {e}")
            continue
        except Exception as e:
            errores_por_anio[year] = f"Error: {repr(e)}"
            print(f"[{year}] ERROR: {e}")
            continue

    print("\n=== Resumen ===")
    print(f"Años OK: {len(ids_por_anio)} -> {sorted(ids_por_anio.keys())}")
    print(f"Años con error: {len(errores_por_anio)} -> {sorted(errores_por_anio.keys())}")

    return ids_por_anio, errores_por_anio


# Resolución automática de `resource_id` (CNMC – CKAN)

## Objetivo
Esta celda implementa un proceso **automático y robusto** para obtener los identificadores
`resource_id` de los datasets **“Precios diarios provinciales”** publicados por la CNMC,
para un rango de años definido por el usuario (**2016–2025**).

El resultado será un **mapa año → resource_id**, que se utilizará posteriormente para
descargar los datos reales de precios desde la API.

---

## Contexto: cómo funciona la API de la CNMC
La plataforma de datos de la CNMC está basada en **CKAN**, que separa el acceso a los datos en dos niveles:

1. **Catálogo (`package_search`)**  
   Permite buscar datasets y obtener metadatos (títulos, recursos, identificadores).
2. **Almacén de datos (`datastore_search`)**  
   Permite acceder a los registros reales, pero requiere conocer previamente el `resource_id`.

Esta celda cubre **exclusivamente el primer nivel**: la resolución automática de los `resource_id`.

---

## Funcionamiento del código
Para cada año del rango indicado:

- Se construye dinámicamente la consulta:
    Precios diarios provinciales - {año} -



- Se consulta el catálogo mediante `package_search`.
- Se selecciona el dataset adecuado:
- Coincidencia exacta por título.
- Coincidencia parcial (el título contiene el texto esperado).
- Fallback al primer resultado si no hay coincidencia clara.
- Se selecciona el `resource_id`:
- Prioridad a recursos con `datastore_active = True`.
- Fallback a `resources[0].id`, validando que funcione con `datastore_search(limit=1)`.
- Si ocurre un error en un año concreto, se registra y el proceso continúa con el siguiente año.

---

## Salida de la función
La función `build_resource_map(start_year, end_year)` devuelve:

- **`ids_por_anio`**  
Diccionario con la forma:
    { año: resource_id }


Contiene únicamente los años resueltos correctamente.

- **`errores_por_anio`**  
Diccionario con los años que fallaron y el motivo del error.

Esto permite separar claramente los años válidos de los que requieren revisión.

---

## Ventajas del enfoque
- Proceso completamente automatizado (sin intervención manual).
- Robusto ante cambios en el orden de los datasets o recursos.
- Compatible con la documentación oficial de CKAN.
- Preparado para reutilizarse en fases posteriores del proyecto (descarga y análisis de datos).

---

> **Nota:**  
> Esta celda **no descarga los datos**. Su única responsabilidad es identificar correctamente
> los `resource_id` necesarios para acceder al DataStore.


In [16]:
import requests
from typing import Dict, Tuple, Any, Optional

CKAN_URL = "https://catalogodatos.cnmc.es"
HEADERS = {"User-Agent": "Application"}

MIN_YEAR = 2016
MAX_YEAR = 2025

# =========================
# CKAN helpers
# =========================
def _package_search(query: str, rows: int = 20) -> Dict[str, Any]:
    """Llama a CKAN package_search y devuelve el JSON (parseado)."""
    url = f"{CKAN_URL}/api/3/action/package_search"
    params = {"q": f'"{query}"', "rows": rows}  # comillas => frase exacta
    r = requests.get(url, headers=HEADERS, params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def _datastore_search_smoke_test(resource_id: str) -> bool:
    """
    Prueba rápida: verifica que el resource_id funciona con datastore_search (limit=1).
    Devuelve True si responde success=True, False si falla (HTTP/CKAN/estructura).
    """
    url = f"{CKAN_URL}/api/3/action/datastore_search"
    params = {"resource_id": resource_id, "limit": 1, "offset": 0}
    try:
        r = requests.get(url, headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        j = r.json()
        return bool(j.get("success", False))
    except Exception:
        return False

# =========================
# Robust picker
# =========================
def _pick_resource_id_robusto(results: list, expected_title: Optional[str] = None) -> Tuple[str, bool]:
    """
    Elige un resource_id de forma robusta.

    Estrategia:
    1) Intentar seleccionar dataset por title (exacto y luego "contiene", case-insensitive)
    2) Si no, usar results[0] (fallback)
    3) Preferir resource con datastore_active=True
    4) Si no hay datastore_active, caer a resources[0].id (fallback web) y marcarlo como fallback=True

    Returns:
    (resource_id, used_fallback)
    """
    if not results:
        raise ValueError("No hubo resultados en package_search (results vacío).")

    dataset = None

    if expected_title:
        # 1) Match exacto por title
        for ds in results:
            if ds.get("title") == expected_title:
                dataset = ds
                break

        # 2) Match por contiene (tolerante)
        if dataset is None:
            expected_lower = expected_title.lower()
            for ds in results:
                title = (ds.get("title") or "").lower()
                if expected_lower in title:
                    dataset = ds
                    break

    # 3) Fallback dataset: el primero
    if dataset is None:
        dataset = results[0]

    resources = dataset.get("resources") or []
    if not resources:
        raise ValueError("Dataset encontrado, pero no tiene resources.")

    # Preferir un resource que esté en DataStore
    for res in resources:
        if res.get("datastore_active") is True and res.get("id"):
            return res["id"], False

    # Fallback al primer recurso, como indica la web
    first_id = resources[0].get("id")
    if first_id:
        return first_id, True

    raise ValueError("No se pudo determinar un resource_id válido (resources[0] sin id).")

# =========================
# Main function
# =========================
def build_resource_map(start_year: int, end_year: int) -> Tuple[Dict[int, str], Dict[int, str]]:
    """
    Construye un diccionario {año: resource_id} para:
    'Precios diarios provinciales - {year} -'
    entre start_year y end_year (incluidos), validando rango [2016, 2025].

    Robustez:
    - Selecciona dataset por title si se puede (expected_title)
    - Prefiere datastore_active=True
    - Si cae a fallback resources[0].id, valida con datastore_search(limit=1):
        - si falla, registra error y salta ese año
    - Si un año falla, lo registra en errores y continúa.

    Devuelve: (ids_por_anio, errores_por_anio)
    """
    # Validaciones del rango
    if not (MIN_YEAR <= start_year <= MAX_YEAR):
        raise ValueError(f"start_year debe estar entre {MIN_YEAR} y {MAX_YEAR}. Recibido: {start_year}")
    if not (MIN_YEAR <= end_year <= MAX_YEAR):
        raise ValueError(f"end_year debe estar entre {MIN_YEAR} y {MAX_YEAR}. Recibido: {end_year}")
    if start_year > end_year:
        raise ValueError(f"start_year ({start_year}) no puede ser mayor que end_year ({end_year}).")

    ids_por_anio: Dict[int, str] = {}
    errores_por_anio: Dict[int, str] = {}

    for year in range(start_year, end_year + 1):
        expected = f"Precios diarios provinciales - {year} -"

        try:
            datos_catalogo = _package_search(query=expected, rows=20)

            if not datos_catalogo.get("success", False):
                errores_por_anio[year] = "CKAN devolvió success=False."
                print(f"[{year}] ERROR: CKAN devolvió success=False")
                continue

            result = datos_catalogo.get("result", {}) or {}
            results = result.get("results", []) or []
            count = result.get("count", 0)

            if count == 0 or not results:
                errores_por_anio[year] = f"No se encontraron datasets con query: {expected}"
                print(f"[{year}] ERROR: No se encontraron resultados (count=0).")
                continue

            resource_id, used_fallback = _pick_resource_id_robusto(results, expected_title=expected)

            # Si usamos fallback (resources[0].id), validamos que sea realmente DataStore (datastore_search funcione)
            if used_fallback:
                ok = _datastore_search_smoke_test(resource_id)
                if not ok:
                    errores_por_anio[year] = (
                        "Se eligió resources[0].id como fallback, pero datastore_search(limit=1) falló. "
                        f"resource_id={resource_id}"
                    )
                    print(f"[{year}] ERROR: fallback resource_id NO funciona con datastore_search -> {resource_id}")
                    continue
                print(f"[{year}] OK (fallback validado) -> resource_id: {resource_id}")
            else:
                print(f"[{year}] OK -> resource_id: {resource_id}")

            ids_por_anio[year] = resource_id

        except requests.exceptions.RequestException as e:
            errores_por_anio[year] = f"Error HTTP/red: {repr(e)}"
            print(f"[{year}] ERROR HTTP/red: {e}")
            continue
        except Exception as e:
            errores_por_anio[year] = f"Error: {repr(e)}"
            print(f"[{year}] ERROR: {e}")
            continue

    print("\n=== Resumen ===")
    print(f"Años OK: {len(ids_por_anio)} -> {sorted(ids_por_anio.keys())}")
    print(f"Años con error: {len(errores_por_anio)} -> {sorted(errores_por_anio.keys())}")

    return ids_por_anio, errores_por_anio





In [ ]:
# ===== Ejemplo de uso rápido =====
ids_por_anio, errores_por_anio = build_resource_map(2016, 2025)
ids_por_anio
errores_por_anio

[2016] OK -> resource_id: a385ec5d-a22b-4029-a322-ce3f40241597
[2017] OK -> resource_id: 4c94e9aa-4973-471c-ae19-6658ec57e865
[2018] OK -> resource_id: e2a074ed-789e-43fc-b7bf-e2ba6106458a
[2019] OK -> resource_id: 898c5d4b-c78b-4653-9226-bc24de59846a
[2020] OK -> resource_id: beb221c5-2be6-472a-bb25-bd6d2343e014
[2021] OK -> resource_id: 9bb7d9fe-b99a-42ea-96f7-35c735b56612
[2022] OK -> resource_id: 42fca586-6582-40c8-8df5-6ebbb8fbfd73
[2023] OK -> resource_id: b5a89db0-239f-4c8a-bd98-6575858359ae
[2024] OK -> resource_id: 141fdb3b-7c56-4eed-bf8d-bee56e577aa6
[2025] OK -> resource_id: 510d138a-6c6d-4dce-8d30-8c77de58d787

=== Resumen ===
Años OK: 10 -> [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Años con error: 0 -> []


{}

In [6]:
resource_id = id_recurso  # el que acabas de obtener

url = f"{CKAN_URL}/api/3/action/datastore_search"
params = {
    "resource_id": resource_id,
    "limit": 740000,   # 5000 es un tamaño típico; luego paginamos si hace falta
    "offset": 0
}

r = requests.get(url, headers=headers, params=params, timeout=60)
r.raise_for_status()

data = r.json()["result"]
records = data["records"]

print("filas recibidas:", len(records))
print("columnas ejemplo:", list(records[0].keys()) if records else "sin registros")
print("primera fila:", records[0] if records else "sin registros")


filas recibidas: 32000
columnas ejemplo: ['_id', 'fecha_precio', 'provincia', 'producto', 'promedio_de_pai_diario_cubo', 'promedio_de_pvp_diario_cubo']
primera fila: {'_id': 1, 'fecha_precio': '2024-01-01', 'provincia': 'Albacete', 'producto': 'Gasolina 95 E5', 'promedio_de_pai_diario_cubo': 0.806, 'promedio_de_pvp_diario_cubo': 1.547}
